In [0]:
%run /Users/scottsbv@gmail.com/student-engagement/utils

In [0]:
%python
# CDC watermark: only read Bronze rows newer than the last email_body write
watermark = get_silver_watermark('lead_activites_email_body')
if watermark:
    print(f"CDC MODE: reading changes after {watermark}")
else:
    print("FULL LOAD: first run")

df = parse_bronze_table('lead_activites_raw', watermark=watermark)
if df is None:
    print(f"✅ CDC: No new rows since {watermark}")
else:
    print(f"body_text_quoted in columns? {'body_text_quoted' in df.columns}")

In [0]:
%python
from pyspark.sql.functions import col

if df is None or "body_text_quoted" not in df.columns:
    # No new Bronze rows, or this batch contains no Emails
    # (type-aware schema sampling then has no email-specific fields)
    df_email_body = None
    print("⏭️ SKIP: no email rows in this batch")
else:
    df_email_body = df.filter(col("type") == "Email") \
        .select(
            col("bronze_insert_date"),
            col("bronze_updated_at"),
            col("id").alias("activity_id"),
            col("lead_id"),
            col("activity_at"),
            col("status"),
            col("date_updated"),
            col("body_text_quoted")[0]["text"].alias("body_text")
        ) \
        .filter(col("body_text").isNotNull())

In [0]:
%python
if df_email_body is not None:
    # Propagate the watermark so write_silver uses CDC (delete + append)
    # instead of full overwrite. First run (watermark=None) stays a full load.
    _CDC_WATERMARK = watermark
    write_silver(df_email_body, "lead_activites_email_body")
    _CDC_WATERMARK = None
else:
    print("⏭️ Nothing to write — table unchanged")

In [0]:
%sql
SELECT
  *
FROM
  crm_ingestion.silver.lead_activites_email_body

WHERE
    lead_id = "lead_00r9MDwFgjZsyfMTSUccUiHxUEN0xuZdRjOH29lOJsN"
AND status = 'inbox'
AND (body_text IS NULL OR body_text NOT ILIKE '%A new event has been scheduled.%')
AND (body_text IS NULL OR body_text NOT ILIKE '%is requesting access to the following folder%')
AND (body_text IS NULL OR body_text NOT ILIKE '%requests access to an item%')
ORDER BY    
    activity_at DESC

In [0]:
display(spark.table('crm_ingestion.silver.lead_activites_email_body').limit(10))